Deep Reinforcement Learning
Assignment 1
Part 1 (MAB - Multi-Armed Bandit)
Group Number - 163

In [2]:
# Installing Required Libraries 
import pandas as pd
import numpy as np
import random 
import matplotlib.pyplot as plt

TASK 1: DATASET DESIGN & ENVIRONMENT SETUP

In [3]:
# Setting seeds for reproducibility - ensures we get the same random numbers
G = 163
random.seed(G)
np.random.seed(G)

In [4]:
# Calculate number of medicines (K = (G % 3) + 5)

K = (G % 3) + 5
print(f"Group Number(G): {G}")
print(f"Number of Medicines(K): {K}")

Group Number(G): 163
Number of Medicines(K): 6


In [11]:
# Calculate hidden success probabilities for each medicine (Pi = 0.4 + ((G + i) % 6) * 0.07)

hidden_probs = []
for i in range(K):
    pi = 0.4 + ((G+i)%6) * 0.07
    hidden_probs.append(pi)

print("Hidden Success Probabilities")
for i,p in enumerate(hidden_probs, start = 1):
    print(f"Medicine: {i}: {p:.2f}")

print(f"\nBest Medicine: Medicine {np.argmax(hidden_probs)+1} "
        f"(Probability = {max(hidden_probs):.2f})")

Hidden Success Probabilities
Medicine: 1: 0.47
Medicine: 2: 0.54
Medicine: 3: 0.61
Medicine: 4: 0.68
Medicine: 5: 0.75
Medicine: 6: 0.40

Best Medicine: Medicine 5 (Probability = 0.75)


In [14]:
# Generating Dataset 
# patient_id: 0 to 999
# severity_score: (patient_id % 5) + 1

num_patients = 1000

# Creating base dataset using pre-known columns
df_base = pd.DataFrame({
    'patient_id': range(num_patients),
    'serverity_score': [(pid % 5)+1 for pid in range(num_patients)]
})

# Adding placeholder columns that algorithms will fill dynamically
df_base['assigned_medicine'] = None
df_base['clinical_outcome']  = None
df_base['utility_score']     = None

# Displaying Dataset Sample
print("Base patient dataset:")
print(df_base.sample(10).to_string(index=False))
print(f"\nTotal patients: {len(df_base)}")
print(f"Severity score range: {df_base['serverity_score'].min()} to "
      f"{df_base['serverity_score'].max()}")


Base patient dataset:
 patient_id  serverity_score assigned_medicine clinical_outcome utility_score
         60                1              None             None          None
        698                4              None             None          None
        416                2              None             None          None
        420                1              None             None          None
        785                1              None             None          None
        911                2              None             None          None
        623                4              None             None          None
        827                3              None             None          None
        346                2              None             None          None
        603                4              None             None          None

Total patients: 1000
Severity score range: 1 to 5


Yugansh Jindal

Priyanshi Singh

In [ ]:

# ε-Greedy Algorithm Implementation
from tqdm import tqdm

class MedicineEnvironment:
    def __init__(self, probabilities):
        self.hidden_probs = probabilities
        self.n_items = len(probabilities)
        self.items = list(range(self.n_items))

    # Randomly sample a patient
    def sample_user(self):
        return np.random.randint(0, num_patients)

    # Reward generation based on hidden probability
    def get_reward(self, item_id, user):
        return 1 if np.random.rand() < self.hidden_probs[item_id] else 0


def run_epsilon_greedy(env, epsilon=0.05, n_visits=20000, n_iterations=20, verbose=True):

    all_runs = []
    total_counts = np.zeros(env.n_items)
    final_Q = np.zeros(env.n_items)

    # Running algorithm multiple times
    for it in tqdm(range(n_iterations), desc=f"Running ε-Greedy Policy (ε={epsilon})"):

        # Estimated reward of each medicine
        Q = np.zeros(env.n_items)

        # Number of times each medicine selected
        N = np.zeros(env.n_items)

        total_reward = 0
        fraction_relevant = np.zeros(n_visits)

        # Simulating patient visits
        for t in range(n_visits):

            # Exploration vs Exploitation
            if np.random.rand() < epsilon:
                arm = np.random.randint(env.n_items)   # Explore
            else:
                arm = np.argmax(Q)                     # Exploit

            # Sample patient
            user = env.sample_user()

            # Select medicine
            item_id = env.items[arm]

            # Get reward
            reward = env.get_reward(item_id, user)

            # Update counts
            N[arm] += 1
            total_counts[arm] += 1

            # Update estimated reward
            Q[arm] += (reward - Q[arm]) / N[arm]

            # Track total reward
            total_reward += reward

            # Running average reward
            fraction_relevant[t] = total_reward / (t + 1)

        final_Q += Q
        all_runs.append(fraction_relevant)

    # Average estimated rewards across all iterations
    final_Q /= n_iterations

    # Print best medicine
    if verbose:
        top_idx = np.argmax(final_Q)
        print(f"\nBest Medicine Found: Medicine {top_idx}")
        print(f"Estimated Success Rate: {final_Q[top_idx]:.3f}")

    return np.mean(all_runs, axis=0), total_counts, final_Q


# Creating Environment
env = MedicineEnvironment(hidden_probs)

# Running ε-Greedy Algorithm
avg_rewards, total_counts, final_Q = run_epsilon_greedy(
    env,
    epsilon=0.05,
    n_visits=20000,
    n_iterations=20
)

# Display results
print("\nFinal Estimated Rewards (Q-values):")
for i, q in enumerate(final_Q):
    print(f"Medicine {i}: {q:.3f}")

print("\nTotal Selection Counts:")
for i, c in enumerate(total_counts):
    print(f"Medicine {i}: {int(c)}")

# Plotting Learning Curve
plt.figure(figsize=(10,5))
plt.plot(avg_rewards)
plt.xlabel("Visits")
plt.ylabel("Average Reward")
plt.title("ε-Greedy Learning Curve")
plt.grid(True)
plt.show()


Niti Jain